In [7]:
import pandas as pd
import numpy as np
import glob
import os

In [ ]:
# df_test = pd.read_csv('data/test_02_Test_2026-08-07_16h16.51.553.csv')
# df_test.head()

In [ ]:
# # Trial-level summary for familiarization and the two test periods

# def last_non_null(series):
#     s = series.dropna()
#     return s.iloc[-1] if len(s) else np.nan


# def first_non_null(series):
#     s = series.dropna()
#     return s.iloc[0] if len(s) else np.nan


# df_work = df_test.copy().reset_index(drop=True)

# # Identify test period index (1, 2) inside each trial from Test_Period.started markers
# is_test_start = df_work['Test_Period.started'].notna()
# df_work['test_period_num'] = is_test_start.groupby(df_work['trial_number']).cumsum()

# # Familiarization summary (one row per trial)
# fam_summary = (
#     df_work.groupby('trial_number', as_index=False)
#     .agg(
#         familiarization_time=('familiarization_time', 'first'),
#         familiarization_time_timeout=('familiarization_time_timeout', 'first'),
#         _fam_started=('Familiarization.started', first_non_null),
#         _fam_stopped=('Familiarization.stopped', first_non_null),
#         gaze_time_ROI_familiarization_last=('gaze_time_ROI_familiarization', last_non_null),
#     )
# )

# # Duration of familiarization period based on explicit start/stop timestamps
# fam_summary['Fam_Duration'] = fam_summary['_fam_stopped'] - fam_summary['_fam_started']
# fam_summary = fam_summary.drop(columns=['_fam_started', '_fam_stopped'])

# # Test period summary (rows for period 1 and 2 in each trial)
# test_rows = df_work[df_work['test_period_num'].between(1, 2)].copy()

# for col in ['gaze_in_roi_left', 'gaze_in_roi_right']:
#     test_rows[col] = pd.to_numeric(test_rows[col], errors='coerce').fillna(0)

# test_summary_long = (
#     test_rows.groupby(['trial_number', 'test_period_num'], as_index=False)
#     .agg(
#         gaze_in_roi_left_count=('gaze_in_roi_left', 'sum'),
#         gaze_in_roi_right_count=('gaze_in_roi_right', 'sum'),
#         gaze_time_in_roi_left_last=('gaze_time_in_roi_left', last_non_null),
#         gaze_time_in_roi_right_last=('gaze_time_in_roi_right', last_non_null),
#     )
# )

# # Convert to wide format: separate columns for test period 1 and 2
# test_summary_wide = test_summary_long.pivot(index='trial_number', columns='test_period_num')
# test_summary_wide.columns = [f'{metric}_test{int(period)}' for metric, period in test_summary_wide.columns]
# test_summary_wide = test_summary_wide.reset_index()

# trial_summary = fam_summary.merge(test_summary_wide, on='trial_number', how='left').sort_values('trial_number')

# print(f"Trials in summary: {trial_summary['trial_number'].nunique()}")
# trial_summary

In [ ]:
# trial_data = pd.DataFrame(columns=['lab_id',	'participant_id',	'familiarization_type',	'trial_num',	'familiarization_condition',	'familiar_stimulus_filename',
# 'novel_stimulus_filename',	'familiarization_stimulus_screen_time',	'familiarization_stimulus_looking_time',	'test_1_novel_stimulus_side',	'test_1_looking_time_novel',
# 'test_1_looking_time_familiar',	'test_1_trial_time',	'test_2_novel_stimulus_side',	'test_2_looking_time_novel',	'test_2_looking_time_familiar',	'test_2_trial_time',
# 'trial_error',	'trial_error_info',	'notes'])

# # lab_id - manual entry
# # partiicipant_id == participant
# # familiarization_type - manual entry
# # trial_num == trial_number
# # familiarization_condition == familiarization_time
# # familiar_stimulus_filename == familiar_stimulus
# # novel_stimulus_filename == novel_stimulus
# # familiarization_stimulus_screen_time == Fam_Duration
# # familiarization_stimulus_looking_time == gaze_time_ROI_familiarization_last
# # test_1_novel_stimulus_side == 'left' if familiar_location_1 == 'right' else 'right'
# # test_1_looking_time_novel == gaze_time_in_roi_left_last_test1 if familiar_location_1 == 'left' else gaze_time_in_roi_right_last_test1
# # test_1_looking_time_familiar == gaze_time_in_roi_right_last_test1 if familiar_location_1 == 'right' else gaze_time_in_roi_left_last_test1
# # test_1_trial_time == ? use Test_Period.started and Test_Period.stopped
# # test_2_novel_stimulus_side == 'left' if familiar_location_2 == 'right' else 'right'
# # test_2_looking_time_novel == gaze_time_in_roi_left_last_test2 if familiar_location_2 == 'right' else gaze_time_in_roi_left_last_test2
# # test_2_looking_time_familiar == gaze_time_in_roi_right_last_test2 if familiar_location_2 == 'right' else gaze_time_in_roi_left_last_test2
# # test_2_trial_time == ? use Test_Period.started and Test_Period.stopped
# # trial_error - manual entry
# # trial_error_info - manual entry
# # notes - manual entry

In [13]:
# Build trial_data (one row per trial) from every data file in a directory
TRIAL_DATA_COLUMNS = ['lab_id', 'participant_id', 'familiarization_type', 'trial_num', 'familiarization_condition',
    'familiar_stimulus_filename', 'novel_stimulus_filename', 'familiarization_stimulus_screen_time',
    'familiarization_stimulus_looking_time', 'test_1_novel_stimulus_side', 'test_1_looking_time_novel',
    'test_1_looking_time_familiar', 'test_1_trial_time', 'test_2_novel_stimulus_side', 'test_2_looking_time_novel',
    'test_2_looking_time_familiar', 'test_2_trial_time', 'trial_error', 'trial_error_info', 'notes']

DURATION_COLUMNS = ['familiarization_stimulus_screen_time', 'familiarization_stimulus_looking_time',
    'test_1_looking_time_novel', 'test_1_looking_time_familiar', 'test_1_trial_time',
    'test_2_looking_time_novel', 'test_2_looking_time_familiar', 'test_2_trial_time']


def _last(series):
    s = series.dropna()
    return s.iloc[-1] if len(s) else np.nan


def _first(series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan


def _with_png(name):
    if pd.isna(name):
        return name
    name = str(name)
    return name if name.lower().endswith('.png') else name + '.png'


def _novel_side(familiar_location):
    if familiar_location == 'left':
        return 'right'
    if familiar_location == 'right':
        return 'left'
    return np.nan


def file_to_trial_data(df):
    """Convert one raw PsychoPy/Tobii data frame (same layout as df_test) to one row per trial."""
    df = df.reset_index(drop=True)
    df = df[df['trial_number'].notna()].copy()
    df['test_period_num'] = df['Test_Period.started'].notna().groupby(df['trial_number']).cumsum()
    for col in ['gaze_time_in_roi_left', 'gaze_time_in_roi_right']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    rows = []
    for trial_number, t in df.groupby('trial_number', sort=True):
        row = dict.fromkeys(TRIAL_DATA_COLUMNS, np.nan)
        row.update(
            participant_id=_first(t['participant']),
            trial_num=trial_number,
            familiarization_condition=_first(t['familiarization_time']),
            familiar_stimulus_filename=_with_png(_first(t['familiar_stimulus'])),
            novel_stimulus_filename=_with_png(_first(t['novel_stimulus'])),
            familiarization_stimulus_screen_time=_first(t['Familiarization.stopped']) - _first(t['Familiarization.started']),
            familiarization_stimulus_looking_time=_last(t['gaze_time_ROI_familiarization']),
            trial_error='no_error',
            trial_error_info='NA',
            notes=_first(t['notes']) if pd.notna(_first(t['notes'])) else 'NA',
        )
        for n in (1, 2):
            fam_loc = _first(t[f'familiar_location_{n}'])
            p = t[t['test_period_num'] == n]
            left, right = _last(p['gaze_time_in_roi_left']), _last(p['gaze_time_in_roi_right'])
            fam_left = fam_loc == 'left'
            row[f'test_{n}_novel_stimulus_side'] = _novel_side(fam_loc)
            row[f'test_{n}_looking_time_novel'] = right if fam_left else left
            row[f'test_{n}_looking_time_familiar'] = left if fam_left else right
            # stop timestamps are logged on a later row than the start, so pair the n-th start with the n-th stop
            starts, stops = t['Test_Period.started'].dropna(), t['Test_Period.stopped'].dropna()
            row[f'test_{n}_trial_time'] = stops.iloc[n - 1] - starts.iloc[n - 1] if min(len(starts), len(stops)) >= n else np.nan
        rows.append(row)
    out = pd.DataFrame(rows, columns=TRIAL_DATA_COLUMNS)
    # raw timings are in seconds; report all measured durations in ms
    out[DURATION_COLUMNS] = out[DURATION_COLUMNS].astype(float) * 1000
    # whole-number columns (nullable Int64 avoids 5.0 / 1.0 in the csv)
    out[['familiarization_condition', 'trial_num']] = out[['familiarization_condition', 'trial_num']].astype('Int64')
    return out


def build_trial_data(data_dir='data', pattern='*.csv', lab_id=np.nan, familiarization_type=np.nan):
    """Run file_to_trial_data over every matching csv in data_dir and stack the results."""
    frames = [file_to_trial_data(pd.read_csv(path, low_memory=False))
              for path in sorted(glob.glob(os.path.join(data_dir, pattern)))]
    if not frames:
        return pd.DataFrame(columns=TRIAL_DATA_COLUMNS)
    out = pd.concat(frames, ignore_index=True)
    # manual-entry columns
    out['lab_id'] = lab_id
    out['familiarization_type'] = familiarization_type
    return out


def prompt_required(label):
    """Ask until a non-empty answer is given."""
    while True:
        answer = input(f'Enter {label}: ').strip()
        if answer:
            return answer
        print(f'{label} cannot be empty.')


def prompt_choice(label, options):
    """Ask until one of the options is chosen, by number or by name."""
    menu = ', '.join(f'{i}) {o}' for i, o in enumerate(options, 1))
    while True:
        answer = input(f'Enter {label} - {menu}: ').strip().strip('"\'').lower()
        if answer.isdigit() and 1 <= int(answer) <= len(options):
            return options[int(answer) - 1]
        if answer in options:
            return answer
        print(f'Please choose one of: {menu}')


def prompt_directory(label='path to the data directory'):
    """Ask until an existing directory containing at least one csv file is given."""
    while True:
        path = os.path.expanduser(input(f'Enter {label}: ').strip().strip('"\''))
        if not os.path.isdir(path):
            print(f'{path!r} is not a directory.')
        elif not glob.glob(os.path.join(path, '*.csv')):
            print(f'No .csv files found in {path!r}.')
        else:
            return path


def prompt_output_path(default_name='trial_data.csv'):
    """Ask for an output csv path (or a directory, which gets default_name); confirm before overwriting."""
    while True:
        path = os.path.expanduser(input(f'Enter output path (.csv file or directory) [{default_name}]: ').strip().strip('"\'') or '.')
        if os.path.isdir(path) or path in ('.', '') or not os.path.splitext(path)[1]:
            path = os.path.join(path, default_name)
        elif not path.lower().endswith('.csv'):
            print('Output file must end in .csv.')
            continue
        if os.path.exists(path) and input(f'{path} exists. Overwrite? [y/N]: ').strip().lower() != 'y':
            continue
        return path


def save_trial_data(df, path):
    """Write df to path as csv, creating parent folders if needed."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    df.to_csv(path, index=False)
    print(f'Saved {len(df)} rows to {path}')


data_dir = prompt_directory()
lab_id = prompt_required('lab_id')
familiarization_type = prompt_choice('familiarization_type', ['infant-controlled', 'fixed-length'])

output_path = prompt_output_path()

trial_data = build_trial_data(data_dir, lab_id=lab_id, familiarization_type=familiarization_type)
save_trial_data(trial_data, output_path)
trial_data

Saved 24 rows to desktop/trial_data.csv


,lab_id,participant_id,familiarization_type,trial_num,familiarization_condition,familiar_stimulus_filename,novel_stimulus_filename,familiarization_stimulus_screen_time,familiarization_stimulus_looking_time,test_1_novel_stimulus_side,test_1_looking_time_novel,test_1_looking_time_familiar,test_1_trial_time,test_2_novel_stimulus_side,test_2_looking_time_novel,test_2_looking_time_familiar,test_2_trial_time,trial_error,trial_error_info,notes
0,elle,test_01,infant-controlled,1,5,Pyramid_Simple.png,Bowtie_Simple.png,5009.1367,5008.4869,right,0.0000,49.9598,5013.7984,left,2733.2868,0.0000,5009.5231,no_error,NA,NA
1,elle,test_01,infant-controlled,2,10,Bowl_Complex.png,Gumdrop_Complex.png,10016.4334,10015.7683,left,2699.9707,150.0757,5016.3106,right,0.0000,4416.6580,5016.2456,no_error,NA,NA
2,elle,test_01,infant-controlled,3,15,SimsDiamond_Complex.png,Cylinder_Complex.png,15003.1555,15002.5846,left,0.0000,3050.0274,5007.7008,right,1949.9578,0.0000,5012.0096,no_error,NA,NA
3,elle,test_01,infant-controlled,4,5,Pacman_Simple.png,Arrow_Simple.png,5003.8545,5003.2392,right,0.0000,4449.9905,5006.3921,left,4433.3430,0.0000,5003.9861,no_error,NA,NA
4,elle,test_01,infant-controlled,5,10,Tripod_Complex.png,Diamond_Complex.png,10011.4433,10010.8710,right,4149.9906,0.0000,5013.8630,left,483.3244,4150.0527,5005.4470,no_error,NA,NA
5,elle,test_01,infant-controlled,6,15,Crinkle_Simple.png,Pumpkin_Simple.png,15008.5572,15007.8857,left,3766.7262,0.0000,5000.4659,right,0.0000,4816.7119,5002.3397,no_error,NA,NA
6,elle,test_01,infant-controlled,7,10,Frac3A_Simple.png,Frac3B_Simple.png,12359.6788,10009.1145,right,1949.8621,433.2928,5013.8374,left,2433.3113,0.0000,5003.9831,no_error,NA,NA
7,elle,test_01,infant-controlled,8,5,Frac1A_Complex.png,Frac1B_Complex.png,5002.2828,5001.7304,left,2033.3567,266.6398,5010.1130,right,4550.0678,212.9362,5013.4797,no_error,NA,NA
8,elle,test_01,infant-controlled,9,15,Frac2A_Complex.png,Frac2B_Complex.png,16870.4332,15003.0902,right,4650.0195,0.0000,5015.0193,left,2466.6275,0.0000,5005.3017,no_error,NA,NA
9,elle,test_01,infant-controlled,10,5,Frac6A_Simple.png,Frac6B_Simple.png,9984.6209,333.9885,right,0.0000,2049.8432,5002.7911,left,0.0000,2716.5974,5000.5722,no_error,NA,NA
